# Fusion des données pour la création de data set

In [1]:
import pandas as pd

## Gestion des fichiers météos

In [36]:
meteo = pd.read_csv("../data/meteo.csv", sep=";")

meteo = meteo[meteo["NUM_POSTE"] == 7240]
meteo = meteo[["DAT", "RR", "TMMOY"]]
meteo = meteo.rename(columns={
    "DAT": "time",
    "RR": "PRELIQ_Q",
    "TMMOY": "T_Q"
})

meteo["time"] = pd.to_datetime(meteo["time"])
meteo["month"] = meteo["time"].dt.to_period("M")

meteo_month = meteo.groupby("month").agg({
    "PRELIQ_Q": "sum",   # pluie mensuelle
    "T_Q": "mean"        # température moyenne mensuelle
}).reset_index()

meteo_month["time"] = meteo_month["month"].dt.to_timestamp()
meteo_month = meteo_month.drop(columns="month")

## Gestion des fichiers ades

In [37]:
nappe = pd.read_csv("../data/nappe.csv", sep=";")

nappe = nappe.rename(columns={
    "date_mesure": "time",
    "niveau_eau_ngf": "niveau_nappe_eau"
})

nappe["time"] = pd.to_datetime(nappe["time"])
nappe["month"] = nappe["time"].dt.to_period("M")

nappe_month = nappe.groupby(["code_bss", "month"]).agg({
    "niveau_nappe_eau": "mean",
    "longitude": "first",
    "latitude": "first"
}).reset_index()

nappe_month["time"] = nappe_month["month"].dt.to_timestamp()
nappe_month = nappe_month.drop(columns="month")

C:\Users\tronn\AppData\Local\Temp\ipykernel_4620\2632561774.py:9: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  nappe["month"] = nappe["time"].dt.to_period("M")


Ou autre fixhier d'ades 

In [39]:
nappe = pd.read_csv("../data/nappe2.txt", sep="|", encoding="latin-1", engine="python")

nappe = nappe.rename(columns={
    "Ancien code national BSS": "code_bss",
    "Date de la mesure": "time",
    "Côte NGF": "niveau_nappe_eau",
    "X_WGS84": "longitude",
    "Y_WGS84": "latitude"
})

nappe["time"] = pd.to_datetime(nappe["time"], dayfirst=True)

nappe["month"] = nappe["time"].dt.to_period("M")

nappe_month = nappe.groupby(["code_bss", "month"]).agg({
    "niveau_nappe_eau": "mean",
    "longitude": "first",
    "latitude": "first"
}).reset_index()

nappe_month["time"] = nappe_month["month"].dt.to_timestamp()

nappe_month = nappe_month.drop(columns="month")

## Gestion des fichiers ETP

In [40]:
etp = pd.read_csv("../data/etp.csv", sep=";")

etp = etp.rename(columns={
    "DATE": "time",
    "ETP_Q_H0175": "ETP_Q",
    "LAMBX": "x",
    "LAMBY": "y"
})

etp["time"] = pd.to_datetime(etp["time"], format="%Y%m%d")
etp["month"] = etp["time"].dt.to_period("M")

etp_month = etp.groupby("month").agg({
    "ETP_Q": "mean",
    "x": "first",
    "y": "first"
}).reset_index()

etp_month["time"] = etp_month["month"].dt.to_timestamp()
etp_month = etp_month.drop(columns="month")

## Fusion

In [41]:
# Filtrer une seule station
nappe_month = nappe_month[
    nappe_month["code_bss"] == "03287X0018/S1"
]

# Fusion mensuelle cohérente
df = pd.merge(meteo_month, etp_month, on="time", how="outer")
df = pd.merge(df, nappe_month, on="time", how="outer")

# Variables finales
df["Peff_All"] = df["PRELIQ_Q"] - df["ETP_Q"]
df.loc[df["Peff_All"] < 0, "Peff_All"] = 0

df = df.sort_values("time")

df = df.rename(columns={
    "longitude": "x",
    "latitude": "y"
})

df = df[[
    "time",
    "x",
    "y",
    "niveau_nappe_eau",
    "PRELIQ_Q",
    "T_Q",
    "ETP_Q",
    "Peff_All",
    "code_bss"
]]

df.insert(0, "id", range(15200, 15200 + len(df)))

print("Doublons temps :", df["time"].duplicated().sum())

df.to_csv("../data/out/dataset_mensuel_complet.csv", sep=";", index=False)

Doublons temps : 0


In [ ]:
https://donneespubliques.meteofrance.fr/?fond=donnee_libre&prefixe=Txt%2FClimat%2Fclimat&extension=csv.gz&date=202601